# Phase 2: Data Cleaning and Preprocessing

**Objective:** Transform raw loan applicant data into a clean, verified dataset ready for feature engineering.

### Key Tasks in this Phase:
1. **Impute Missing Values:** Handle the ~3.3% missing records in `Employment.Type`.
2. **Data Sanity & Outlier Validation:** Confirm no negative or corrupt loan amounts or asset costs.
3. **Drop Non-Predictive Identifiers:** Remove primary keys (`UniqueID`) and arbitrary high-cardinality IDs (`Employee_code_ID`, `supplier_id`, `Current_pincode_ID`) to prevent overfitting.
4. **Export Clean Baseline:** Save cleaned data to `data/train_cleaned.csv`.

### Step 2.1: Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

# Load raw training dataset
TRAIN_PATH = Path("../data/train.csv") if Path("../data/train.csv").exists() else Path("data/train.csv")
df = pd.read_csv(TRAIN_PATH)

print(f"Raw dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

### Step 2.2: Handle Missing Values in `Employment.Type`

> **Viva Insight:** Rather than dropping rows or guessing between salaried and self-employed, we impute missing records with a distinct category `'Missing'`. This preserves all 233,154 rows and allows the model to learn if non-disclosure correlates with default.

In [ ]:
print("--- Before Imputation ---")
print(df['Employment.Type'].value_counts(dropna=False))

# Impute missing values with 'Missing'
df['Employment.Type'] = df['Employment.Type'].fillna('Missing')

print("\n--- After Imputation ---")
print(df['Employment.Type'].value_counts(dropna=False))

# Verify zero missing values remain in entire dataset
remaining_nulls = df.isnull().sum().sum()
print(f"\nTotal remaining NaN values in dataset: {remaining_nulls}")
assert remaining_nulls == 0, "There are still missing values!"

### Step 2.3: Data Sanity Checks (Financial Ranges & Boundaries)

Validate that loan disbursed amounts, asset costs, and Loan-to-Value (LTV) ratios are positive and mathematically valid.

In [ ]:
# Sanity check 1: Negative or zero loan amounts
invalid_disbursed = (df['disbursed_amount'] <= 0).sum()
invalid_asset = (df['asset_cost'] <= 0).sum()
invalid_ltv = ((df['ltv'] <= 0) | (df['ltv'] > 100)).sum()

print("Data Sanity Audit:")
print(f"  - Invalid Disbursed Amounts (<=0): {invalid_disbursed}")
print(f"  - Invalid Asset Costs (<=0):       {invalid_asset}")
print(f"  - Invalid LTV (<=0% or >100%):     {invalid_ltv}")
print(f"  - LTV Min: {df['ltv'].min():.2f}% | LTV Max: {df['ltv'].max():.2f}%")

### Step 2.4: Drop Non-Predictive & Leaky Identifiers

> **Viva Insight:** Columns like `UniqueID` (applicant database ID), `Employee_code_ID`, `supplier_id`, and `Current_pincode_ID` are high-cardinality IDs. If fed into tree models or neural networks, the model will memorize specific IDs rather than learning general credit risk patterns (overfitting).

In [ ]:
# Columns to drop: arbitrary database keys and extreme high-cardinality noise IDs
cols_to_drop = ['UniqueID', 'supplier_id', 'Current_pincode_ID', 'Employee_code_ID']

print(f"Columns before dropping: {df.shape[1]}")
df_cleaned = df.drop(columns=cols_to_drop)
print(f"Columns after dropping {cols_to_drop}: {df_cleaned.shape[1]}")

### Step 2.5: Verify and Export Cleaned Dataset

Save the cleaned baseline to `data/train_cleaned.csv` for Phase 3 (Feature Engineering).

In [ ]:
OUTPUT_PATH = Path("../data/train_cleaned.csv") if Path("../data").exists() else Path("data/train_cleaned.csv")
df_cleaned.to_csv(OUTPUT_PATH, index=False)
print(f"Successfully exported cleaned dataset to: {OUTPUT_PATH.resolve()}")
print(f"Final Clean Shape: {df_cleaned.shape[0]:,} rows x {df_cleaned.shape[1]} columns")
df_cleaned.head(2)